# 📡 Telco Customer Churn Prediction: Random Forest vs XGBoost

Welcome! Retaining customers is much cheaper than acquiring new ones. In the telecommunications industry, predicting which customers are highly likely to leave (churn) is a multi-million dollar data science problem.

This end-to-end project demonstrates how to identify at-risk customers by comparing two powerful machine learning algorithms: **Random Forest** and **XGBoost**.

### 🔍 Project Highlights:
* **Data Preprocessing:** Handling missing values, cleaning numeric formats, and label encoding categorical features.
* **Predictive Modeling:** Training and evaluating Random Forest and XGBoost classifiers.
* **Business Intelligence Integration:** Transforming raw test predictions into an interactive dashboard for stakeholders.

### 📊 Interactive Business Dashboard
I have built an interactive dashboard based on the test set predictions. Stakeholders can use this to filter customer segments (e.g., by Contract type) and compare the churn predictions visually.

👉 **[Click Here to Explore the Interactive Looker Studio Dashboard](https://datastudio.google.com/reporting/463f0d3e-93e4-45f2-a208-5f05125f65db)**

---
*Feel free to explore the code, fork this notebook, or connect with me for feedback and collaboration opportunities!*

In [1]:
import pandas as pd

# Paste file path yang tadi Anda copy ke dalam tanda kutip di bawah ini:
file_path = '/kaggle/input/datasets/hanggersukmapd/analisis-prediksi-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
# Membaca data CSV ke dalam variabel bernama 'df' (Data Frame)
df = pd.read_csv(file_path)

# Menampilkan 5 data teratas
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# 1. Membuang kolom customerID karena tidak memiliki pola prediksi
df = df.drop('customerID', axis=1)

# 2. Mengubah kolom TotalCharges menjadi angka numerik 
# (opsi errors='coerce' menyulap baris spasi menjadi nilai kosong/NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 3. Mengisi nilai yang kosong (NaN) tersebut dengan angka 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 4. Mengecek struktur data terbaru untuk memastikan semuanya aman
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [3]:
from sklearn.preprocessing import LabelEncoder

# Membuat alat penerjemah teks ke angka
le = LabelEncoder()

# Mencari semua kolom yang bertipe teks (object) lalu mengubahnya menjadi angka secara otomatis
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

# Menampilkan 5 data teratas untuk melihat perubahannya
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [4]:
from sklearn.model_selection import train_test_split

# 1. Memisahkan 'Target' (yang mau ditebak) dengan 'Fitur' (bahan tebakan)
X = df.drop('Churn', axis=1)
y = df['Churn']

# 2. Membelah data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Menampilkan jumlah baris data setelah dibelah
print("Jumlah data latih (Training):", len(X_train))
print("Jumlah data uji (Testing):", len(X_test))

Jumlah data latih (Training): 5634
Jumlah data uji (Testing): 1409


In [5]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 1. Menyiapkan 'mesin' algoritma
rf_model = RandomForestClassifier(random_state=42)
xgb_model = XGBClassifier(random_state=42)

# 2. Melatih kedua model (proses belajar)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

# 3. Menyuruh model menebak data uji (proses ujian)
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

# 4. Menghitung dan menampilkan nilai ujian (akurasi)
print("Akurasi Random Forest : {:.2f}%".format(accuracy_score(y_test, rf_pred) * 100))
print("Akurasi XGBoost       : {:.2f}%".format(accuracy_score(y_test, xgb_pred) * 100))

Akurasi Random Forest : 79.49%
Akurasi XGBoost       : 78.99%


In [6]:
# 1. Menyatukan data uji dengan jawaban asli dan hasil tebakan
hasil_prediksi = X_test.copy()
hasil_prediksi['Target_Asli'] = y_test
hasil_prediksi['Tebakan_Random_Forest'] = rf_pred
hasil_prediksi['Tebakan_XGBoost'] = xgb_pred

# 2. Menyimpan data menjadi file CSV untuk Looker Studio
nama_file = 'hasil_prediksi_churn_rf_vs_xgb.csv'
hasil_prediksi.to_csv(nama_file, index=False)

print(f"Bagus! File {nama_file} berhasil disimpan dan siap diunduh.")
# Menampilkan cuplikan hasil akhir
hasil_prediksi[['Target_Asli', 'Tebakan_Random_Forest', 'Tebakan_XGBoost']].head()

Bagus! File hasil_prediksi_churn_rf_vs_xgb.csv berhasil disimpan dan siap diunduh.


,Target_Asli,Tebakan_Random_Forest,Tebakan_XGBoost
185,1,1,1
2715,0,0,0
3825,0,0,0
1807,1,1,1
132,0,0,0
